![Universidad Central](https://universidad.ucentral.edu.co/tulengua/wp-content/themes/tulengua/images/logo-ucentral.png)

<h2 align="center">🎛️ Orquestador del Proyecto — CRISP-DM</h2>

## Estimación del gasto personal — Tesis de Maestría

Este notebook permite ejecutar **cada fase del proyecto de forma independiente**, sin necesidad de correr todo el pipeline de nuevo. Cada fase lee sus insumos desde archivos `.parquet` (guardados al final de la fase anterior) y exporta sus resultados al directorio `02_intermedios/`.

### Arquitectura de persistencia

```
BASE_PATH/
├── 01_datos/raw/              ← Microdatos ENPH (zip + dataset/)
├── 02_intermedios/            ← Archivos parquet entre fases
│   ├── raw_*.parquet          ← DataFrames fuente (NB01 → NB02)
│   ├── nb02_*.parquet         ← DataFrames limpios (NB02 → NB03)
│   ├── nb03_df_temp_a.parquet ← Dataset analítico (NB03 → NB04)
│   └── nb04_df_modelado.parquet ← Dataset modelado (NB04 → NB05/06)
├── produccion/modelos/        ← Artefactos .pkl finales
│   ├── artefactos_modelo_v2.pkl
│   └── artefactos_modelo_hibrido.pkl
└── notebooks/
    ├── 00_orquestador.ipynb   ← ESTE ARCHIVO
    ├── 01_carga_datos.ipynb
    ├── 02_limpieza_variables.ipynb
    ├── 03_universo_analitico.ipynb
    ├── 04_modelado.ipynb
    ├── 05_modelo_hibrido.ipynb
    └── 06_evaluacion_implementacion.ipynb
```

### Estudiantes:
- Oscar Leonardo Duarte Urrego  
- Paola Andrea Velandia Lozano  

### Director de Tesis:
- Miguel Hernández Bejarano


In [8]:

# ============================================================
# CONFIGURACIÓN DEL ORQUESTADOR
# ============================================================
from pathlib import Path
import subprocess, sys, os, time

# Instalar papermill si no está disponible
try:
    import papermill as pm
    print("✅ papermill disponible")
except ImportError:
    print("📦 Instalando papermill...")
    subprocess.run([sys.executable, "-m", "pip", "install", "papermill", "-q"], check=True)
    import papermill as pm
    print("✅ papermill instalado")

# Rutas
NOTEBOOKS_DIR = Path.cwd()  # Ajusta si los notebooks están en otra carpeta
BASE_PATH = NOTEBOOKS_DIR.parent if NOTEBOOKS_DIR.name in ("notebooks", "notebook") else NOTEBOOKS_DIR

NOTEBOOKS = {
    "NB01": NOTEBOOKS_DIR / "01_carga_datos.ipynb",
    "NB02": NOTEBOOKS_DIR / "02_limpieza_variables.ipynb",
    "NB03": NOTEBOOKS_DIR / "03_universo_analitico.ipynb",
    "NB04": NOTEBOOKS_DIR / "04_modelado.ipynb",
    "NB05": NOTEBOOKS_DIR / "05_modelo_hibrido.ipynb",
    "NB06": NOTEBOOKS_DIR / "06_evaluacion_implementacion.ipynb",
}

OUTPUT_DIR_EXEC = BASE_PATH / "03_ejecutados"
OUTPUT_DIR_EXEC.mkdir(exist_ok=True)

print(f"📁 Base del proyecto : {BASE_PATH}")
print(f"📁 Notebooks         : {NOTEBOOKS_DIR}")
print(f"📁 Ejecutados        : {OUTPUT_DIR_EXEC}")
print()
for k, v in NOTEBOOKS.items():
    estado = "✅" if v.exists() else "❌ NO ENCONTRADO"
    print(f"  {estado} {k}: {v.name}")


✅ papermill disponible
📁 Base del proyecto : /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4
📁 Notebooks         : /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/notebooks
📁 Ejecutados        : /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados

  ✅ NB01: 01_carga_datos.ipynb
  ✅ NB02: 02_limpieza_variables.ipynb
  ✅ NB03: 03_universo_analitico.ipynb
  ✅ NB04: 04_modelado.ipynb
  ✅ NB05: 05_modelo_hibrido.ipynb
  ✅ NB06: 06_evaluacion_implementacion.ipynb


In [9]:

# ============================================================
# FUNCIÓN DE EJECUCIÓN
# ============================================================
def ejecutar_notebook(nb_key: str, params: dict = None):
    """
    Ejecuta un notebook individualmente con papermill.
    
    Args:
        nb_key: 'NB01'...'NB06'
        params: diccionario de parámetros opcionales a inyectar
    """
    nb_path = NOTEBOOKS[nb_key]
    out_path = OUTPUT_DIR_EXEC / nb_path.name
    
    if not nb_path.exists():
        raise FileNotFoundError(f"No se encontró {nb_path}")
    
    print(f"\n{'='*60}")
    print(f"▶️  Ejecutando {nb_key}: {nb_path.name}")
    print(f"   Output: {out_path}")
    print(f"{'='*60}")
    
    t0 = time.time()
    try:
        pm.execute_notebook(
            str(nb_path),
            str(out_path),
            parameters=params or {},
            kernel_name="python3",
            progress_bar=True,
        )
        elapsed = time.time() - t0
        print(f"\n✅ {nb_key} completado en {elapsed:.1f}s")
        return True
    except pm.exceptions.PapermillExecutionError as e:
        print(f"\n❌ Error en {nb_key}: {e}")
        print(f"   Revisa el notebook ejecutado en: {out_path}")
        return False


def ejecutar_pipeline_completo():
    """Ejecuta todos los notebooks en secuencia. Se detiene si uno falla."""
    print("🚀 Iniciando pipeline completo CRISP-DM\n")
    t0 = time.time()
    
    for nb_key in ["NB01", "NB02", "NB03", "NB04", "NB05", "NB06"]:
        ok = ejecutar_notebook(nb_key)
        if not ok:
            print(f"\n⛔ Pipeline detenido en {nb_key}. Corrige el error y vuelve a ejecutar solo esa fase.")
            return False
    
    print(f"\n🎉 Pipeline completo en {(time.time()-t0)/60:.1f} min")
    return True

print("✅ Funciones del orquestador listas")
print()
print("USO:")
print("  ejecutar_notebook('NB01')          → Ejecuta solo carga de datos")
print("  ejecutar_notebook('NB03')          → Ejecuta solo universo analítico")
print("  ejecutar_pipeline_completo()       → Ejecuta todo en secuencia")


✅ Funciones del orquestador listas

USO:
  ejecutar_notebook('NB01')          → Ejecuta solo carga de datos
  ejecutar_notebook('NB03')          → Ejecuta solo universo analítico
  ejecutar_pipeline_completo()       → Ejecuta todo en secuencia


---
## 🔍 Estado del pipeline

In [10]:

# ============================================================
# VERIFICAR ESTADO DE LOS ARCHIVOS INTERMEDIOS
# ============================================================
from pathlib import Path
import os

PERSIST_DIR = BASE_PATH / "02_intermedios"

print(f"📁 {PERSIST_DIR}\n")
if not PERSIST_DIR.exists():
    print("⚠️  El directorio no existe aún — ejecuta NB01 primero.")
else:
    archivos = sorted(PERSIST_DIR.glob("*.parquet"))
    if not archivos:
        print("⚠️  Sin archivos intermedios — ejecuta los notebooks en orden.")
    else:
        for f in archivos:
            size_mb = f.stat().st_size / 1_048_576
            print(f"  ✅ {f.name:<55} {size_mb:.1f} MB")

print()
# Verificar artefactos pkl
for pkl_name in ['artefactos_modelo_v2.pkl', 'artefactos_modelo_hibrido.pkl']:
    p = Path.cwd() / pkl_name
    if p.exists():
        size_mb = p.stat().st_size / 1_048_576
        print(f"  ✅ {pkl_name:<55} {size_mb:.1f} MB")
    else:
        print(f"  ❌ {pkl_name} — no generado aún")


📁 /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/02_intermedios

  ✅ nb02_AGRUP_04_GmfMP.parquet                             0.9 MB
  ✅ nb02_AGRUP_08_GmfMP.parquet                             0.8 MB
  ✅ nb02_GmenosFreqUrbano_homo_objetivo_def.parquet         0.7 MB
  ✅ nb02_agrup_GdComidasfueraH_objetivo_hogar.parquet       0.4 MB
  ✅ nb02_agrup_GdComidasfueraH_objetivo_perso.parquet       0.2 MB
  ✅ nb02_df_GdUrbano.parquet                                0.9 MB
  ✅ nb02_df_resumen_GPersonComidasfueraH_objetivo_def.parquet 0.4 MB
  ✅ nb02_gdpersonales_homo_objetivo_def.parquet             0.7 MB
  ✅ nb02_poblacion_objetivo_final.parquet                   2.4 MB
  ✅ nb02_publico_objetivo_5.parquet                         2.1 MB
  ✅ nb03_df_temp_a.parquet                                  7.9 MB
  ✅ nb04_df_modelado.parquet                                13.0 MB
  ✅ raw_caracteristicas_generales_personas.parquet          14.0 MB
  ✅ raw_cargue_inflacion.parquet                           

---
## 🩺 Dashboard de Volumetrías y Diagnóstico del Pipeline

Ejecuta la celda siguiente para obtener un **reporte completo del estado de cada fase**, incluyendo:

- Número de registros y columnas en cada dataset intermedio
- Población expandida (ΣFex_C) por fase
- Presencia y rango de variables clave (ingresos, gastos, log-transformaciones)
- Distribución de clústeres KMeans e híbridos
- Métricas del modelo WLS (R², RMSE) y clasificador (balanced accuracy)
- Diagnóstico de validez del modelo híbrido
- Resumen ejecutivo con semáforo ✅ / ⚠️ / ❌ por notebook

> Esta celda **solo lee** archivos ya generados — no ejecuta ningún notebook ni modifica datos.


In [11]:

# ============================================================
# DIAGNÓSTICO DE VOLUMETRÍAS — TODAS LAS FASES
# Lee los parquets intermedios y .pkl para validar el pipeline
# sin necesidad de ejecutar ningún notebook.
# ============================================================
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import os, traceback, datetime

PERSIST_DIR = BASE_PATH / "02_intermedios"

# ── Paleta de colores ANSI ────────────────────────────────────────────────────
OK   = "[92m✅[0m"
WARN = "[93m⚠️ [0m"
ERR  = "[91m❌[0m"
HDR  = "[1;96m"
RST  = "[0m"
SEP  = "─" * 72

def fmt_n(n):   return f"{int(n):>12,}"
def fmt_pct(p): return f"{p:>7.1f}%"
def fmt_r2(r):  return f"{r:>8.4f}"

def check(cond, msg_ok, msg_fail):
    return f"  {OK} {msg_ok}" if cond else f"  {WARN} {msg_fail}"

def load_parquet(name):
    p = PERSIST_DIR / name
    if not p.exists():
        return None, f"archivo no encontrado: {p}"
    try:
        df = pd.read_parquet(p)
        return df, None
    except Exception as e:
        return None, str(e)

print(f"\n{HDR}{'='*72}{RST}")
print(f"{HDR}  DASHBOARD DE VOLUMETRÍAS — PIPELINE CRISP-DM TESIS{RST}")
print(f"{HDR}  Generado: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}{RST}")
print(f"{HDR}{'='*72}{RST}")
print(f"  Directorio intermedios : {PERSIST_DIR}")
print()

resultados_globales = {}   # para el resumen final

# ════════════════════════════════════════════════════════════════════════════════
# FASE 1 — CARGA DE MICRODATOS
# ════════════════════════════════════════════════════════════════════════════════
print(f"\n{HDR}[NB01] FASE 1 — CARGA DE MICRODATOS{RST}")
print(SEP)

raw_files = sorted(PERSIST_DIR.glob("raw_*.parquet")) if PERSIST_DIR.exists() else []
n_raw = len(raw_files)
tablas_clave = [
    "raw_caracteristicas_generales_personas",
    "raw_viviendas_y_hogares",
    "raw_gastos_diarios_urbanos",
    "raw_gastos_menos_frecuentes_-_articulos",
    "raw_gastos_diarios_personales_urbano",
    "raw_cargue_inflacion",
]

if n_raw == 0:
    print(f"  {ERR} Sin archivos raw_*.parquet — ejecuta NB01 primero.")
    resultados_globales["NB01"] = {"status": "❌ No ejecutado", "n_raw": 0}
else:
    print(f"  {OK} DataFrames fuente cargados: {n_raw}")
    for fname in tablas_clave:
        p = PERSIST_DIR / f"{fname}.parquet"
        if p.exists():
            df_tmp = pd.read_parquet(p)
            size_mb = p.stat().st_size / 1_048_576
            print(f"      {OK} {fname.replace('raw_',''):<55} {fmt_n(len(df_tmp))} filas | {size_mb:.1f} MB")
        else:
            print(f"      {ERR} {fname.replace('raw_',''):<55} NO ENCONTRADO")
    resultados_globales["NB01"] = {"status": f"✅ {n_raw} tablas", "n_raw": n_raw}


# ════════════════════════════════════════════════════════════════════════════════
# FASE 2 — LIMPIEZA Y VARIABLES ANALÍTICAS
# ════════════════════════════════════════════════════════════════════════════════
print(f"\n{HDR}[NB02] FASE 2 — LIMPIEZA, HOMOLOGACIÓN Y VARIABLES ANALÍTICAS{RST}")
print(SEP)

nb02_exports = {
    "publico_objetivo_5":                         "Público objetivo final (≥1 SMMLV)",
    "df_GdUrbano":                                "Gastos diarios urbanos + Cap.C (agrupados)",
    "GmenosFreqUrbano_homo_objetivo_def":          "Gastos menos frecuentes (homologados)",
    "gdpersonales_homo_objetivo_def":             "Gastos diarios personales (homologados)",
    "agrup_GdComidasfueraH_objetivo_hogar":       "Comidas fuera hogar — nivel hogar",
    "agrup_GdComidasfueraH_objetivo_perso":       "Comidas fuera hogar — nivel persona",
    "df_resumen_GPersonComidasfueraH_objetivo_def": "Gastos personales comidas fuera hogar",
    "AGRUP_04_GmfMP":                             "Gastos menos frecuentes MP cód.04",
    "AGRUP_08_GmfMP":                             "Gastos menos frecuentes MP cód.08",
    "poblacion_objetivo_final":                   "Población objetivo final (alias)",
}

nb02_ok = True
pub_obj_shape = None
for fname, label in nb02_exports.items():
    df_tmp, err = load_parquet(f"nb02_{fname}.parquet")
    if err:
        print(f"  {ERR} {label:<55} {err}")
        nb02_ok = False
    else:
        n_rows, n_cols = df_tmp.shape
        if fname == "publico_objetivo_5":
            pub_obj_shape = df_tmp.shape
            # Auditar FEX_C
            fex = pd.to_numeric(df_tmp["FEX_C"].astype(str).str.replace(",","."), errors="coerce")
            pob_expandida = fex.sum()
            pct_nulos = df_tmp.isnull().mean().mean() * 100
            print(f"  {OK} {label:<55} {fmt_n(n_rows)} filas | {n_cols} cols")
            print(f"       Población expandida (ΣFex_C): {pob_expandida:>18,.0f} personas")
            print(f"       Nulos promedio               : {pct_nulos:>7.2f}%")
            print(check(pct_nulos < 10, "Calidad OK (<10% nulos)", f"Alta tasa de nulos: {pct_nulos:.1f}%"))
        else:
            print(f"  {OK} {label:<55} {fmt_n(n_rows)} filas | {n_cols} cols")

if nb02_ok and pub_obj_shape:
    resultados_globales["NB02"] = {"status": f"✅ {pub_obj_shape[0]:,} personas público objetivo", "shape": pub_obj_shape}
else:
    resultados_globales["NB02"] = {"status": "❌ Incompleto o no ejecutado"}


# ════════════════════════════════════════════════════════════════════════════════
# FASE 3 — UNIVERSO ANALÍTICO Y VARIABLES ECONÓMICAS
# ════════════════════════════════════════════════════════════════════════════════
print(f"\n{HDR}[NB03] FASE 3 — UNIVERSO ANALÍTICO Y VARIABLES ECONÓMICAS{RST}")
print(SEP)

df_nb03, err_nb03 = load_parquet("nb03_df_temp_a.parquet")
if err_nb03:
    print(f"  {ERR} Dataset analítico no encontrado: {err_nb03}")
    resultados_globales["NB03"] = {"status": "❌ No ejecutado"}
else:
    n_rows, n_cols = df_nb03.shape
    print(f"  {OK} Dataset analítico (df_temp_a): {fmt_n(n_rows)} filas | {n_cols} columnas")

    # Variables económicas clave
    vars_economicas = {
        "INGRESOS_AL_2025" : "Ingresos actualizados 2025 (COP)",
        "GASTOS_AL_2025"   : "Gastos actualizados 2025 (COP)",
        "log_ingresos_2025": "Log ingreso 2025",
        "log_gastos_2025"  : "Log gasto 2025",
        "ratio_gastos_2025": "Ratio gasto/ingreso",
        "FEX_C"            : "Factor de expansión FEX_C",
    }
    print()
    print("  Variables económicas clave:")
    for col, label in vars_economicas.items():
        if col in df_nb03.columns:
            serie = pd.to_numeric(df_nb03[col].astype(str).str.replace(",","."), errors="coerce")
            nulos = serie.isna().sum()
            pct_n = nulos / len(df_nb03) * 100
            estado = OK if pct_n < 5 else WARN
            print(f"  {estado} {label:<45} media={serie.mean():>12,.1f}  nulos={pct_n:.1f}%")
        else:
            print(f"  {ERR} {label:<45} COLUMNA AUSENTE")

    # Variables categoricas clave
    vars_cat = ["actividad_ppal", "nivel_educ_agrupado", "Estrato", "antiguedad_agrup", "Sexo_"]
    print()
    print("  Distribución variables categóricas clave:")
    for col in vars_cat:
        if col in df_nb03.columns:
            n_cats = df_nb03[col].nunique()
            nulos  = df_nb03[col].isna().mean() * 100
            print(f"  {OK} {col:<30} {n_cats:>3} categorías | nulos={nulos:.1f}%")
        else:
            print(f"  {WARN} {col:<30} columna ausente")

    fex = pd.to_numeric(df_nb03["FEX_C"].astype(str).str.replace(",","."), errors="coerce") if "FEX_C" in df_nb03.columns else pd.Series([])
    resultados_globales["NB03"] = {
        "status": f"✅ {n_rows:,} × {n_cols} cols",
        "shape": (n_rows, n_cols),
        "pob_expandida": fex.sum() if len(fex) > 0 else 0
    }


# ════════════════════════════════════════════════════════════════════════════════
# FASE 4 — MODELADO WLS + KMEANS
# ════════════════════════════════════════════════════════════════════════════════
print(f"\n{HDR}[NB04] FASE 4 — MODELADO WLS + SEGMENTACIÓN KMEANS{RST}")
print(SEP)

df_nb04, err_nb04 = load_parquet("nb04_df_modelado.parquet")
art_v2_path = Path.cwd() / "artefactos_modelo_v2.pkl"

if err_nb04:
    print(f"  {ERR} Dataset modelado no encontrado: {err_nb04}")
    resultados_globales["NB04"] = {"status": "❌ No ejecutado"}
else:
    n_rows, n_cols = df_nb04.shape
    print(f"  {OK} Dataset modelado (df_temp_a): {fmt_n(n_rows)} filas | {n_cols} columnas")

    # Verificar columnas producidas por el modelado
    cols_modelo = {
        "log_gasto_pred_w"    : "Predicción WLS (log)",
        "residuo_log_gasto_w" : "Residuo WLS",
        "cluster_final"       : "Asignación KMeans (gasto estimado)",
        "gasto_estimado"      : "Gasto estimado (escala original)",
        "ratio_estimado"      : "Ratio estimado gasto/ingreso",
        "FEX_C_norm"          : "FEX_C normalizado",
    }
    print()
    print("  Columnas producidas por el modelo:")
    for col, label in cols_modelo.items():
        if col in df_nb04.columns:
            serie = pd.to_numeric(df_nb04[col].astype(str).str.replace(",","."), errors="coerce")
            nulos = serie.isna().mean() * 100
            estado = OK if nulos < 5 else (WARN if nulos < 20 else ERR)
            print(f"  {estado} {label:<45} nulos={nulos:.1f}%  rango=[{serie.min():.3f}, {serie.max():.3f}]")
        else:
            print(f"  {ERR} {label:<45} COLUMNA AUSENTE")

    # Distribución de clusters
    if "cluster_final" in df_nb04.columns:
        dist = df_nb04["cluster_final"].value_counts().sort_index()
        print()
        print("  Distribución KMeans (cluster_final):")
        for k, n in dist.items():
            bar = "█" * int(n / dist.max() * 20)
            pct = n / len(df_nb04) * 100
            print(f"    Clúster {int(k)}: {fmt_n(n)} ({pct:.1f}%) {bar}")

    # Artefacto pkl v2
    print()
    if art_v2_path.exists():
        try:
            art_v2 = joblib.load(art_v2_path)
            r2_test = art_v2.get("r2_test", "n/a")
            rmse    = art_v2.get("rmse_test", "n/a")
            n_clust = art_v2.get("n_clusters", "n/a")
            n_keys  = len(art_v2)
            size_mb = art_v2_path.stat().st_size / 1_048_576
            print(f"  {OK} artefactos_modelo_v2.pkl  ({size_mb:.1f} MB | {n_keys} claves)")
            _r2_ok   = isinstance(r2_test, float) and not (r2_test != r2_test)  # not nan
            _rmse_ok = isinstance(rmse,    float) and not (rmse    != rmse)
            if _r2_ok:   print(f"       R² test WLS    : {fmt_r2(r2_test)}")
            else:         print(f"       R² test WLS    : {r2_test} (no disponible en pkl v2)")
            if _rmse_ok: print(f"       RMSE test (log): {fmt_r2(rmse)}")
            print(f"       Clústeres KM   : {n_clust}")
            if _r2_ok:
                print(check(r2_test > 0.25,
                    f"R² = {r2_test:.4f} ≥ umbral (0.25)",
                    f"R² = {r2_test:.4f} bajo (umbral: 0.25)"))
            else:
                print(f"  {WARN} R² no disponible en pkl v2 — ver métricas en pkl híbrido (NB05)")
        except Exception as e:
            print(f"  {ERR} Error cargando artefactos_modelo_v2.pkl: {e}")
    else:
        print(f"  {ERR} artefactos_modelo_v2.pkl no encontrado — ejecuta NB04")

    resultados_globales["NB04"] = {
        "status": f"✅ {n_rows:,} × {n_cols} cols",
        "r2_test": art_v2.get("r2_test") if art_v2_path.exists() else None
    }


# ════════════════════════════════════════════════════════════════════════════════
# FASE 4b — MODELO HÍBRIDO
# ════════════════════════════════════════════════════════════════════════════════
print(f"\n{HDR}[NB05] FASE 4b — MODELO HÍBRIDO (RESIDUOS WLS + CLASIFICADOR){RST}")
print(SEP)

art_h_path = Path.cwd() / "artefactos_modelo_hibrido.pkl"
if not art_h_path.exists():
    print(f"  {ERR} artefactos_modelo_hibrido.pkl no encontrado — ejecuta NB05")
    resultados_globales["NB05"] = {"status": "❌ No ejecutado"}
else:
    try:
        art_h = joblib.load(art_h_path)
        n_keys = len(art_h)
        size_mb = art_h_path.stat().st_size / 1_048_576

        print(f"  {OK} artefactos_modelo_hibrido.pkl  ({size_mb:.1f} MB | {n_keys} claves)")

        claves_esperadas = [
            "model_wls", "kmeans_hibrido", "clf_pipeline",
            "ratio_table_hibrido", "cluster_map_hibrido"
        ]
        print()
        print("  Claves del artefacto:")
        for k in claves_esperadas:
            estado = OK if k in art_h else ERR
            print(f"  {estado} {k}")

        # Métricas del clasificador
        acc_clf  = art_h.get("acc_clf_test",           "n/a")
        bal_acc  = art_h.get("cv_balanced_acc_mean",   "n/a")
        r2_test  = art_h.get("r2_test",                "n/a")
        n_clust  = art_h.get("n_clusters",             "n/a")
        print()
        print("  Métricas del modelo híbrido:")
        _r2_h_ok = isinstance(r2_test, float) and not (r2_test != r2_test)
        if _r2_h_ok:
            print(f"    {OK} WLS R² test            : {fmt_r2(r2_test)}")
        else:
            # Try to get r2 from the v2 pkl if available
            _r2_from_h = art_h.get('r2_test')
            if isinstance(_r2_from_h, float) and not (_r2_from_h != _r2_from_h):
                print(f"    {OK} WLS R² test (pkl híbrido): {fmt_r2(_r2_from_h)}")
            else:
                print(f"    {WARN} WLS R² test            : no disponible (NaN)")
        if isinstance(acc_clf, float): print(f"    {OK} Clasificador accuracy   : {fmt_r2(acc_clf)}")
        if isinstance(bal_acc, float):
            estado = OK if bal_acc >= 0.65 else (WARN if bal_acc >= 0.45 else ERR)
            print(f"    {estado} Balanced accuracy CV     : {fmt_r2(bal_acc)}")
            print(check(bal_acc >= 0.65, "Balanced acc ≥ 0.65 → modelo híbrido válido",
                        f"Balanced acc = {bal_acc:.4f} (umbral: 0.65)"))

        # Clusters híbridos desde el parquet
        if df_nb04 is not None and "cluster_hibrido" in df_nb04.columns:
            print()
            print("  Distribución clústeres híbridos (cluster_hibrido):")
            dist_h = df_nb04["cluster_hibrido"].value_counts().sort_index()
            for k, n in dist_h.items():
                bar = "█" * int(n / dist_h.max() * 20)
                pct = n / len(df_nb04) * 100
                print(f"    Clúster {int(k)}: {fmt_n(n)} ({pct:.1f}%) {bar}")

        resultados_globales["NB05"] = {
            "status": f"✅ bal_acc={bal_acc:.4f}" if isinstance(bal_acc, float) else "✅ Ejecutado",
            "bal_acc": bal_acc
        }
    except Exception as e:
        print(f"  {ERR} Error cargando artefacto híbrido: {e}")
        resultados_globales["NB05"] = {"status": "❌ Error al cargar pkl"}


# ════════════════════════════════════════════════════════════════════════════════
# FASE 5-6 — EVALUACIÓN E IMPLEMENTACIÓN
# ════════════════════════════════════════════════════════════════════════════════
print(f"\n{HDR}[NB06] FASES 5-6 — EVALUACIÓN E IMPLEMENTACIÓN{RST}")
print(SEP)

if not art_h_path.exists():
    print(f"  {ERR} Requiere artefactos_modelo_hibrido.pkl — ejecuta NB05 primero")
    resultados_globales["NB06"] = {"status": "❌ Dependencia faltante"}
else:
    try:
        art_h = joblib.load(art_h_path)
        diag  = art_h.get("diagnostico_hibrido", None)

        if diag is None:
            print(f"  {WARN} diagnostico_hibrido no encontrado en el artefacto.")
            print(f"       Ejecuta NB06 (H5 + H4b) para agregar el diagnóstico.")
            resultados_globales["NB06"] = {"status": "⚠️ Diagnóstico pendiente"}
        else:
            decision    = diag.get("decision",           "n/a")
            bal_acc_h   = diag.get("bal_acc_hibrido",    "n/a")
            eta2        = diag.get("eta2_residuo_cluster","n/a")
            sep_res     = diag.get("separacion_residuo", "n/a")
            n_ok        = diag.get("n_criterios_ok",     "n/a")
            pct_may     = diag.get("pct_cluster_mayoritario", "n/a")

            print(f"  {OK} Diagnóstico de validez disponible")
            print()
            print(f"  {'Balanced Accuracy clasificador':<45}: {bal_acc_h}")
            print(f"  {'Eta² residuo-clúster (≥0.10=válido)':<45}: {eta2}")
            print(f"  {'Separación P50 residuo (≥0.30=OK)':<45}: {sep_res}")
            print(f"  {'% clúster mayoritario (<60%=OK)':<45}: {pct_may}")
            print(f"  {'Criterios cumplidos':<45}: {n_ok} / 5")
            print()

            if "HÍBRIDO" in str(decision).upper():
                print(f"  {OK} DECISIÓN: {decision}")
            elif "SIMPLE" in str(decision).upper():
                print(f"  {WARN} DECISIÓN: {decision}")
            else:
                print(f"  {WARN} DECISIÓN: {decision}")

            resultados_globales["NB06"] = {
                "status": f"✅ {decision}",
                "criterios_ok": n_ok
            }
    except Exception as e:
        print(f"  {ERR} Error al leer diagnóstico: {e}")
        resultados_globales["NB06"] = {"status": "❌ Error"}


# ════════════════════════════════════════════════════════════════════════════════
# RESUMEN EJECUTIVO
# ════════════════════════════════════════════════════════════════════════════════
print(f"\n{HDR}{'='*72}{RST}")
print(f"{HDR}  RESUMEN EJECUTIVO DEL PIPELINE{RST}")
print(f"{HDR}{'='*72}{RST}")

estado_iconos = {"✅": OK, "❌": ERR, "⚠️": WARN}

nb_labels = {
    "NB01": "Carga microdatos",
    "NB02": "Limpieza y variables analíticas",
    "NB03": "Universo analítico y vars económicas",
    "NB04": "Modelado WLS + KMeans",
    "NB05": "Modelo híbrido",
    "NB06": "Evaluación + Implementación",
}

for nb_key, label in nb_labels.items():
    info   = resultados_globales.get(nb_key, {"status": "⚪ Sin información"})
    status = info.get("status", "⚪ Sin información")
    prefix = OK if "✅" in status else (ERR if "❌" in status else WARN)
    print(f"  {prefix} {nb_key} — {label:<40} {status}")

print()
print(f"  Directorio intermedios : {PERSIST_DIR}")
print(f"  Última actualización   : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  {SEP}")



  DASHBOARD DE VOLUMETRÍAS — PIPELINE CRISP-DM TESIS
  Generado: 2026-05-03 19:19:55
  Directorio intermedios : /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/02_intermedios


[NB01] FASE 1 — CARGA DE MICRODATOS
────────────────────────────────────────────────────────────────────────
  ✅ DataFrames fuente cargados: 14
      ✅ caracteristicas_generales_personas                           291,590 filas | 14.0 MB
      ✅ viviendas_y_hogares                                           87,201 filas | 7.4 MB
      ✅ gastos_diarios_urbanos                                     3,393,520 filas | 21.1 MB
      ✅ gastos_menos_frecuentes_-_articulos                        2,111,331 filas | 10.9 MB
      ✅ gastos_diarios_personales_urbano                             524,948 filas | 3.9 MB
      ✅ cargue_inflacion                                               2,448 filas | 0.0 MB

[NB02] FASE 2 — LIMPIEZA, HOMOLOGACIÓN Y VARIABLES ANALÍTICAS
────────────────────────────────────────────────────────────

---
## 🔧 Ejecución por fases individuales
> Ejecuta **solo la celda de la fase que necesites reprocesar**. Cada fase leerá automáticamente sus insumos desde los parquets guardados.

## ▶️ Ejecutar Fase 1 — Carga de Microdatos
**Cuándo re-ejecutar:** cuando cambie la fuente de datos (nuevo ZIP, nuevo FILE_ID en GDrive).

In [12]:
ejecutar_notebook("NB01")


▶️  Ejecutando NB01: 01_carga_datos.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/01_carga_datos.ipynb


/Users/paolavelandia/anaconda3/lib/python3.10/site-packages/nbformat/__init__.py:93: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Executing:   0%|          | 0/14 [00:00<?, ?cell/s]


✅ NB01 completado en 39.4s


True

## ▶️ Ejecutar Fase 2 — Limpieza y Variables Analíticas
**Cuándo re-ejecutar:** cuando modifiques reglas de limpieza, variables de ingreso, subsidios, homologaciones.

In [13]:
ejecutar_notebook("NB02")


▶️  Ejecutando NB02: 02_limpieza_variables.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/02_limpieza_variables.ipynb


Executing:   0%|          | 0/72 [00:00<?, ?cell/s]


✅ NB02 completado en 47.1s


True

## ▶️ Ejecutar Fase 3 — Universo Analítico y Variables Económicas
**Cuándo re-ejecutar:** cuando cambies la integración de fuentes, indexación IPC, o el análisis descriptivo.

In [14]:
ejecutar_notebook("NB03")


▶️  Ejecutando NB03: 03_universo_analitico.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/03_universo_analitico.ipynb


Executing:   0%|          | 0/40 [00:00<?, ?cell/s]


✅ NB03 completado en 39.3s


True

## ▶️ Ejecutar Fase 4 — Modelado WLS + KMeans
**Cuándo re-ejecutar:** cuando cambies la selección de variables, hiperparámetros del modelo o número de clusters.

In [15]:
ejecutar_notebook("NB04")


▶️  Ejecutando NB04: 04_modelado.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/04_modelado.ipynb


Executing:   0%|          | 0/18 [00:00<?, ?cell/s]


✅ NB04 completado en 57.9s


True

## ▶️ Ejecutar Fase 4b — Modelo Híbrido
**Cuándo re-ejecutar:** cuando cambies el clasificador de producción, la lógica de residuos o los artefactos del modelo.

In [16]:
ejecutar_notebook("NB05")


▶️  Ejecutando NB05: 05_modelo_hibrido.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/05_modelo_hibrido.ipynb


Executing:   0%|          | 0/15 [00:00<?, ?cell/s]


✅ NB05 completado en 37.3s


True

## ▶️ Ejecutar Fases 5-6 — Evaluación e Implementación
**Cuándo re-ejecutar:** cuando quieras re-evaluar el modelo o simular nuevos casos de producción.

In [17]:
ejecutar_notebook("NB06")


▶️  Ejecutando NB06: 06_evaluacion_implementacion.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/06_evaluacion_implementacion.ipynb


Executing:   0%|          | 0/21 [00:00<?, ?cell/s]


✅ NB06 completado en 4.8s


True

---
## ▶️ Ejecutar Pipeline Completo
Ejecuta todos los notebooks en secuencia. Si alguno falla, el pipeline se detiene y muestra el error.

In [18]:
ejecutar_pipeline_completo()

🚀 Iniciando pipeline completo CRISP-DM


▶️  Ejecutando NB01: 01_carga_datos.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/01_carga_datos.ipynb


Executing:   0%|          | 0/14 [00:00<?, ?cell/s]


✅ NB01 completado en 37.0s

▶️  Ejecutando NB02: 02_limpieza_variables.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/02_limpieza_variables.ipynb


Executing:   0%|          | 0/72 [00:00<?, ?cell/s]


✅ NB02 completado en 45.5s

▶️  Ejecutando NB03: 03_universo_analitico.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/03_universo_analitico.ipynb


...

Executing:   0%|          | 0/15 [00:00<?, ?cell/s]


✅ NB05 completado en 40.7s

▶️  Ejecutando NB06: 06_evaluacion_implementacion.ipynb
   Output: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/03_ejecutados/06_evaluacion_implementacion.ipynb


Executing:   0%|          | 0/21 [00:00<?, ?cell/s]


✅ NB06 completado en 6.3s

🎉 Pipeline completo en 3.8 min


True

In [ ]:
# cd /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/notebooks/02_app
# streamlit run app.py

grep "Empalado\|Empleado" /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/notebooks/02_app/app.py